# Scope

> The approach of edgeslm, and the decisions we make as we refine it

edgeslm fine-tunes small language models (SLMs) for narrow tasks and runs them on edge hardware. This page describes the approach through its first task: short WhatsApp messages that tell coffee farmers which plants are short of water and which sensors need fixing. It is a working document, and its sections change as we refine the approach.

## Problem and framing

A large hosted model can already do an expert's analysis. In the [batch 2 diagnosis](https://fr.anckalbi.net/speaking-plants/batches/b02/diagnosis.html) of speaking-plants, Claude found the water-stressed plant from the sensor logs, much as a plant physiologist would. Using such a model for every message in the field has costs:

- every call costs money and energy,
- farm data leaves the farm,
- the service depends on a provider, and on the network at the moment of the call,
- a narrow, repeated task uses only a small part of the model's expertise.

edgeslm takes the other route: a small model, fine-tuned for one narrow task, that runs on a board on the farm. The model is like a chip that holds one slice of domain expertise. The pipeline that builds it is domain-neutral: a task is defined by data, namely its input format, its output format and its examples.

### The first task

The first task comes from [speaking-plants](https://fr.anckalbi.net/speaking-plants/), which detects water stress in coffee plants with thermal cameras, soil moisture sensors and an air sensor. Sensors are costly to run, so only some plants carry a sensor node: a farm can have dozens of nodes, each standing for the plants around it. The task has two parts:

- **Triage, in code.** Feature engineering and a classical ML model decide, for one farm and one day, which nodes to watch or fix, how severe each case is, and what the data shows about its cause, such as whether water still reaches the soil. This part belongs to the domain project, not to edgeslm.
- **The message, by the SLM.** The model turns the triage into a short WhatsApp message to the farmer.

Deciding what is critical is the code's job. Saying it clearly, with an action, is the model's job.

The message follows these rules:

- It covers only the nodes to watch or fix. When nothing is critical, nothing is sent.
- It says what to do, such as watering or maintenance. It gives no measurements or counts: a value such as a leaf-to-air temperature difference means little to a farmer as an absolute number, and the engineered features and the ML model already capture the signal. A count of flagged nodes would mislead too, because only some plants are monitored and a problem on a shared line usually reaches the plants between them.
- It covers two kinds of problem: water stress, and equipment faults such as a failed sensor or a silent node.
- Each item is one paragraph: its severity marker, its location exactly as in the triage, a colon, then what is wrong and what to do. The markers are 👀 early, ⚠️ moderate and 🚨 severe. Later mentions use words such as "the line" or "its dripper", not the location again.
- Its paragraphs follow the triage's order, most severe first, and say when each problem started with the triage's phrase, such as since today, since Tuesday or for a week or more. It never gives a date.
- It is at most 600 characters long, so that it fits on a phone screen.
- It is sent in the evening or early in the morning, before irrigation. Water stress shows most clearly in the afternoon.

Here is a sketch of one triage output and its message, for an invented farm with 40 sensor nodes. The triage contract, below, defines its fields and rules:

```json
{
  "farm": "fazenda-exemplo", "date": "2026-10-08", "nodes": 40,
  "items": [
    {"problem": "water stress", "location": "Block B, drip line B2", "pattern": "cluster",
     "nodes_flagged": 6, "nodes_in_group": 8, "severity": "moderate", "since": "since Tuesday",
     "soil_water": "none since Tuesday"},
    {"problem": "equipment fault", "location": "Block A, row 1", "pattern": "one node",
     "severity": "moderate", "since": "since Monday", "fault": "missing data"},
    {"problem": "water stress", "location": "Block C, row 3", "pattern": "one node",
     "severity": "early", "since": "since today", "soil_water": "arriving"}
  ]
}
```

> ⚠️ Block B, drip line B2: most monitored plants have been short of water since Tuesday, and no water has reached their soil since then. Check the line today, from its start to the last plant.
>
> ⚠️ Block A, row 1: the sensors at this plant have sent no data since Monday, so it is not monitored. Check the node's power and connection.
>
> 👀 Block C, row 3: a monitored plant has shown first signs of water stress since today, although its soil still gets water. Check the plant for heat damage or root problems.

### Language

The farmers in the first task read Brazilian Portuguese. Small models are usually weaker in Portuguese than in English, so the baseline measures candidate models in Portuguese before any fine-tuning. Fine-tuning should help: it picks up style and vocabulary from little data, and regional farm vocabulary is that kind of target. If small models still fall short, a fixed template can carry the locations, markers and actions, and the model writes only the rest. The proof of concept uses English.

### Hardware

The model runs with llama.cpp on the gateway, the board that collects the nodes' data over LoRa radio. Two boards are candidates. [Rovai compared both](https://mjrovai.com/articles/slm-edge-boards-compared/) with the same models and the same llama.cpp build:

| | Raspberry Pi 5 | Arduino UNO Q |
|---|---|---|
| CPU | 4 × Cortex-A76 | 4 × Cortex-A53, plus an STM32 microcontroller |
| RAM | 8 GB | 4 GB |
| Speed limited by | memory bandwidth | compute |
| Price, September 2026 | about $211 with cooler | $79 |

Messages are generated once a day per farm, so speed is not the main limit. On the UNO Q, Qwen3.5 2B generated 2.53 tokens per second, so a message of about 100 tokens takes under a minute. Memory sets the limit instead: by our estimate, about 4B parameters on the Pi 5 and about 2B on the UNO Q. The choice of board follows from the evaluation: the smallest model that meets the quality threshold decides the cheapest board that can run it.

Sending a WhatsApp message needs a network connection, so a hosted model would often be reachable too. The case for a local model rests on the other reasons: data stays on the farm, there is no cost per message, energy use is low, nothing depends on a provider, and generation does not wait for the network, because the gateway can queue the message until the connection is back.

## Pipeline

Building a model for a task follows these stages:

1. **Synthetic triage inputs**, sampled across the task's dimensions, as the next section describes.
2. **Teacher messages:** a large model, Claude, writes the message for each input.
3. **Baseline:** candidate base models write messages for the test inputs before any fine-tuning.
4. **Fine-tuning:** LoRA adapters trained on pairs of inputs and messages. On Apple Silicon this uses MLX; on CUDA GPUs, tools such as unsloth do the same job.
5. **Evaluation** of the baseline and fine-tuned models on the same held-out test set.
6. **Export:** the adapter is fused into the base model and converted to GGUF, the single-file format that llama.cpp loads.
7. **Deployment** on the target board with llama.cpp, and a final evaluation there.

Domain experts improve the data at stages 1, 2 and 5.

Over time, the two kinds of model take different roles. A large model is the teacher during development, and handles new kinds of cases when a connection is available. The small model handles the repeated task in the field.

### Two teachers

Two models write the teacher messages: Claude Haiku and Claude Sonnet. Sonnet's judgement of actions and urgency was somewhat better on a first batch of ten records, at about three times the cost, and ten records cannot settle the choice. Each message records the model that wrote it, the temperature, the version of the prompt and the date, so that a comparison only ever pairs messages written with the same prompt.

Both models write a message for every training input. Experts review a subset of about 50 inputs in both versions, in random order and without knowing which model wrote which message, so that the review shows whether the difference in judgement matters to them. Fine-tuning the SLM at each training size on one teacher's messages and then on the other's gives two learning curves. They show whether a better teacher gives a better student. If the difference is small, the cheaper teacher is enough.

## Synthetic data generation

Synthetic inputs follow the method of [synthetic questions for MARIS](https://fr.anckalbi.net/posts/synthetic-questions-maris/): find the dimensions along which cases vary, give each a few levels, and sample combinations of levels, called tuples. Each tuple stands for a distinct case the model must handle. A few dimensions keep every tuple covered. Ten dimensions with a few levels each would give tens of thousands of tuples, most of them never sampled.

The first task has three dimensions with three levels each, so 27 tuples:

| Dimension | Levels |
|---|---|
| Problem | water stress, equipment fault, both |
| Spatial pattern | one node, a cluster sharing a line or block, widespread |
| Severity | early, moderate, severe |

The levels mean different things for the two kinds of problem:

- **Spatial pattern.** For water stress, one node points to a single plant or dripper, a cluster to a shared irrigation line, and widespread stress to the weather or the irrigation schedule. `soil_water` tells these apart: when water still arrives, the weather is the likelier cause. For equipment faults, one node points to that node, a cluster to what its nodes share, such as their power supply, and widespread faults to the gateway.
- **Severity.** For water stress, it is the plant's state: first signs (early), stress for several days (moderate), or a risk of damage (severe). For equipment faults, it is how long and how badly data is lost, whatever the spatial pattern: data missing for hours, implausible values or a changed camera view (early), no data for 1 to 3 days (moderate), or no data for more than 3 days (severe).

A node that sends no readings cannot be flagged for water stress, as the triage contract below requires, because there is no data to detect stress from. So when a `both` record has a widespread pattern and a moderate or severe severity, the widespread item is the water stress, and the fault is a single node or a cluster. With this rule all 27 tuples are valid, and a widespread, severe fault is a farm left unmonitored for days.

Other variation changes the details of a message but not its kind, so it is drawn at random within each tuple rather than made a dimension: farm size, how locations are named (blocks and rows, numbered plots, greenhouse benches), how long the problem has lasted, and the number of items.

The test set has 10 inputs per tuple, 270 in total, and no test input is ever used for training. Training sets are sampled across all tuples, so scores can be broken down by tuple to show which cases a model gets wrong.

## Triage contract

The proof of concept replaces the domain project's triage with generated records, from [`01_coffee`](../01_coffee.ipynb). The generator is a mock, but every rule it follows to keep records consistent is also a requirement on the real triage. This section lists the fields and rules, so that the domain project and edgeslm agree on what the triage delivers. The generator is the executable form of this contract: its tests check every rule on every record of the test set.

### Fields

One record covers one farm and one day:

| Field | Meaning |
|---|---|
| `farm`, `date` | The farm, and the day of the triage. |
| `nodes` | The number of sensor nodes on the farm. |
| `items` | One entry per problem to watch or fix, most severe first. |

Each item has these fields:

| Field | Meaning |
|---|---|
| `problem` | `water stress` or `equipment fault`. |
| `location` | A place the farmer knows: one node, a group of nodes on a shared irrigation line, or the whole farm. |
| `pattern` | `one node`, `cluster` or `widespread`. |
| `severity` | `early`, `moderate` or `severe`, as defined under Synthetic data generation. |
| `since` | When the problem started, as a phrase the message can use as it is: since today, since yesterday, since a weekday within the past week, or for a week or more. |
| `soil_water` | For water stress only: whether water still reaches the soil, from irrigation or rain. `arriving` means it does, so more water may not help. Otherwise a phrase such as `none since Tuesday` means the soil is drying, so water is not reaching the plants. |
| `fault` | For equipment faults only: what the data shows. `missing data` means a node sends nothing, `implausible values` means a sensor reports values no plant or weather could produce, such as a soil moisture drop too fast for uptake, and `camera view changed` means the thermal camera no longer sees the same leaves. |
| `sensor` | For `implausible values` only: `soil moisture`, `thermal camera`, `air temperature` or `humidity`. |
| `nodes_flagged`, `nodes_in_group` | For clusters and widespread items only: how many nodes of the group are flagged, out of how many. |
| `amount` | For clusters and widespread items only: the word for the share of flagged nodes, which the message uses as it is: `all`, `almost all` (80% or more), `most` (more than half) or `some` (half or fewer). |

### Rules

- **Readable locations.** Locations are names the farmer uses, such as a block and drip line, a row, a plot or a bench. They are never node IDs.
- **Groups follow irrigation.** A cluster is a group of nodes on one irrigation line or valve, so that its action, such as checking the line, reaches all the plants it covers.
- **One location, one item.** No two items point at the same node or group.
- **Most severe first.** Items are sorted by severity, most severe first, so that the message can keep their order.
- **Aggregation.** When one kind of problem flags more than 70% of the farm's nodes, the triage reports one widespread item with one cause, not several local ones. The single nodes and clusters of one kind stay below that threshold.
- **No stress without data.** A node that sends no readings is reported as a fault, and is never flagged for water stress.
- **Observed causes.** A water stress item says through `soil_water` whether water reaches the soil, which the soil sensors show directly as a rise after irrigation or rain. The message gives the action at the level the data supports, such as checking the line, and never names a part that no sensor watches, such as a valve. An equipment fault item says through `fault` what the data shows, and `sensor` names the sensor when values are implausible. Its action says where to look, by pattern: one node, that node; several nodes, what they share, such as the gateway or the power supply.
- **No raw values.** Weather, readings and engineered features stay inside the ML model. They reach the triage only through severity, `since`, `soil_water`, `fault` and `sensor`. Raw values in the triage would invite the SLM to reason about them, which is the code's job.

Some rules need more than sensor data. Readable locations need a map from each node to its place on the farm. Groups need the irrigation layout of each farm. `soil_water` needs no labels, because the soil sensors show whether water arrives: the [batch 2 diagnosis](https://fr.anckalbi.net/speaking-plants/batches/b02/diagnosis.html) found that watering had stopped from the missing morning rise in soil moisture. Faults need no labels either: `fault` records what the data shows, and never a cause no sensor watches, such as an empty battery or a lost radio link. The node map and the irrigation layout are the main open questions for the design of the ML side.

## Domain expert annotators

Synthetic data and a teacher model are a start. Domain knowledge enters through experts, who:

- audit the dimensions and tuples, and say which cases are missing or unrealistic,
- review and edit synthetic messages: an edited message becomes better training data, and the difference from the draft shows where the teacher's judgement is weak,
- do error analysis on model outputs, by writing a free-text note on what is wrong (open coding) and then grouping the notes into failure modes (axial coding),
- label outputs as pass or fail for each failure mode, to align the judges described under Evaluation,
- write messages for real cases, in the same format.

Experts start from drafts rather than blank pages, which is faster and gives more consistent messages. For the first task, they must read and write Brazilian Portuguese.

The dataset of expert-reviewed examples is the lasting artifact, not the model. Each model is rebuilt from a version of the dataset, so it is always possible to say whose expertise, from which version of the data, went into which model.

A minimal annotation app, built with FastHTML and storing its data in SQLite, supports this work. It shows one triage input next to its message. The expert can edit the message, mark it as pass or fail, and write a note.

## Evaluation

Evaluation decides whether a model is good enough, and so which board it needs. Three rules hold throughout:

- every model, including the untuned base model, is scored on the same held-out test set,
- once expert-reviewed examples exist, the test set is weighted towards them, because synthetic data can train a model but should not be its final judge,
- final scores are measured on the target board.

### Checks computed by code

Some failure modes can be checked by comparing a message with its triage input, without any labels:

- **Format:** every paragraph starts with a marker and a location, followed by a colon.
- **Coverage:** every item in the input has its paragraph.
- **No invented items:** every paragraph names a location from the input, and no location has two paragraphs.
- **Severity:** each paragraph has the marker of its item's severity, 👀, ⚠️ or 🚨.
- **No numbers:** the message contains no digits outside its locations, so no readings, counts or dates. Numbers that are part of a location, such as row 3, are allowed.
- **Length:** the message is at most 600 characters long.
- **Amount:** each cluster or widespread item uses its record's `amount` word.

### Judges for the other failure modes

Other failure modes need judgement, such as whether the action fits what the triage reports. They follow the method of [LLM as a judge](https://fr.anckalbi.net/posts/llm-as-a-judge/):

1. Error analysis by experts finds the failure modes.
2. Each failure mode gets its own judge, a large model that labels a message as pass or fail for that one failure mode.
3. Expert labels are split into a training set (10 to 20%, for few-shot examples in the judge's prompt), a development set (40 to 45%, to refine the prompt) and a test set (40 to 45%). No example moves between sets.
4. On the test set, the judge's true positive rate (TPR, the share of good messages it passes) and true negative rate (TNR, the share of bad messages it fails) are measured against the expert labels.
5. The judge's observed pass rate on new messages is corrected with the Rogan-Gladen formula, which needs TPR + TNR > 1:

$$\theta = \frac{\text{observed rate} + \text{TNR} - 1}{\text{TPR} + \text{TNR} - 1}$$

6. A bootstrap over the test set gives a 95% confidence interval for the corrected rate.

## Proof of concept

The proof of concept is for a meeting, and has four aims:

1. Explain the setup and why: the use case, and the problem it addresses.
2. Show that an SLM can be fine-tuned for the task, and how much data it needs.
3. Explain the role of domain experts: auditing and improving synthetic data, annotating new data, and error analysis for the judges.
4. Show why AI at the edge suits this task: hosted LLM inference costs money and energy, farm data is sensitive, and the task needs only a small part of a large model's expertise.

It uses synthetic data only, in English. The expert phase is explained rather than run, and the annotation app shows how it would work. The steps:

1. **Schema and sampler:** the triage JSON schema, the 27 tuples, and random variation within each.
2. **Teacher messages:** Claude Haiku and Claude Sonnet each write the message for every sampled input, as described under Two teachers, and a sample is checked by hand.
3. **Baseline:** candidate base models of about 2B parameters or smaller, so that they run on the UNO Q, are scored on the 270 test inputs.
4. **Metric:** the checks computed by code, broken down by tuple.
5. **Fine-tuning:** LoRA with MLX on Apple Silicon, trained on 50, 100, 200, 400 and 800 examples, once with each teacher's messages. The scores against the number of examples form a learning curve for each teacher, which answers how much data the task needs and whether a better teacher gives a better student.
6. **Export and edge demo:** the best model is converted to GGUF, run with llama.cpp, and handed to a colleague with an Arduino UNO Q, with example inputs, the prompt and the command, to measure speed and memory on the board.
7. **Annotation app:** the minimal FastHTML app, shown on a few synthetic examples.
8. **Presentation**, built from the results of steps 3 to 7.

## Terms

- **SLM:** small language model, here with up to about 4 billion parameters.
- **LoRA:** Low-Rank Adaptation, a fine-tuning method that trains small adapter matrices next to the frozen weights of a model. Not to be confused with **LoRa**, the long-range radio that links the nodes to the gateway. In code, we avoid a bare `lora` name.
- **GGUF:** the single-file model format of llama.cpp, holding the weights, usually quantized, with the tokenizer and metadata.
- **Node:** a sensor station on one plant, which stands for the plants around it. Only some plants carry one.
- **Triage:** the output of the domain code, listing the nodes to watch or fix for one farm and one day.
- **Tuple:** one combination of levels, one from each dimension.
- **Teacher:** the large model that writes reference messages.
- **Failure mode:** one specific way a message can be wrong, found by error analysis.